In [1]:
import pandapower as pp
def test_installation():
    return pp.__version__
version = test_installation()
print(version)

3.2.1


In [7]:
import pandapower as pp

def create_simple_network():
    net = pp.create_empty_network()

    bus_1 = pp.create_bus(net, vn_kv=20.0, name="Bus source")
    bus_2 = pp.create_bus(net, vn_kv=20.0, name="Bus load")

    pp.create_ext_grid(net, bus=bus_1, vm_pu=1.0, name="External grid")

    pp.create_line(
        net,
        from_bus=bus_1,
        to_bus=bus_2,
        length_km=2.5,
        std_type="NAYY 4x50 SE",
        name="Line 1"
    )

    pp.create_load(
        net,
        bus=bus_2,
        p_mw=1.0,
        q_mvar=0.2,
        name="Load 1"
    )

    return net


def run_power_flow(net):
    pp.runpp(net)
    return net


net = create_simple_network()
net = run_power_flow(net)

print(net.res_bus)
print(net.res_line)


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


      vm_pu  va_degree      p_mw    q_mvar
0  1.000000   0.000000 -1.004159 -0.134835
1  0.995884   0.008772  1.000000  0.200000
   p_from_mw  q_from_mvar  p_to_mw  q_to_mvar     pl_mw   ql_mvar  i_from_ka  \
0   1.004159     0.134835     -1.0       -0.2  0.004159 -0.065165   0.029248   

    i_to_ka      i_ka  vm_from_pu  va_from_degree  vm_to_pu  va_to_degree  \
0  0.029561  0.029561         1.0             0.0  0.995884      0.008772   

   loading_percent  
0        20.817523  


In [8]:
net = pp.create_empty_network()

line_data = pp.load_std_type(
    net,
    name="NAYY 4x50 SE",
    element="line"
)

line_data


{'c_nf_per_km': 210,
 'r_ohm_per_km': 0.642,
 'x_ohm_per_km': 0.083,
 'max_i_ka': 0.142,
 'type': 'cs',
 'q_mm2': 50,
 'alpha': 0.00403,
 'voltage_rating': 'LV'}

In [7]:
import pandas as pd

xls = pd.ExcelFile("/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/0_0_grid.xlsx")
print(xls.sheet_names)

['bus', 'load', 'ext_grid', 'line', 'trafo', 'bus_geodata', 'line_std_types', 'trafo_std_types', 'trafo3w_std_types', 'fuse_std_types', 'res_bus', 'res_line', 'res_trafo', 'res_ext_grid', 'res_load', 'dtypes', 'parameters']


In [9]:
import pandas as pd
import pandapower as pp
file_path = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/1_0_grid.xlsx"
lv_grid=pp.from_excel(file_path)
pp.runpp(lv_grid)

KeyError: 'parameter'

In [ ]:
import pandas as pd
import pandapower as pp
import traceback

file_path = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/1_0_grid.xlsx"

def diagnose_pandapower_excel(filename):
    print(f"Fichier analyse: {filename}\n")

    try:
        xls = pd.read_excel(filename, sheet_name=None, index_col=0, engine="openpyxl")
    except Exception:
        print("Impossible de lire le fichier Excel avec pandas:")
        traceback.print_exc()
        return

    print("Feuilles trouvees:")
    for sheet_name in xls.keys():
        print(f"- {repr(sheet_name)}")

    print("\nVerification de bus_geodata:")
    if "bus_geodata" not in xls:
        print("La feuille 'bus_geodata' n'est pas trouvee exactement avec ce nom.")
        similar = [name for name in xls.keys() if "bus" in str(name).lower() or "geo" in str(name).lower()]
        print("Feuilles proches:", similar)
    else:
        bg = xls["bus_geodata"]
        print("Apercu de bus_geodata:")
        display(bg.head(10))
        print("Colonnes lues par pandas:", list(bg.columns))
        print("Nom de l'index:", repr(bg.index.name))
        print("Premiers index:", list(bg.index[:10]))

        missing_columns = [col for col in ["x", "y"] if col not in bg.columns]
        if missing_columns:
            print("\nProbleme probable: colonnes manquantes:", missing_columns)
            print("Si x/y sont visibles dans Excel mais pas ici, l'en-tete n'est probablement pas au bon rang.")
        else:
            print("\nOK: les colonnes x et y sont bien lues.")

        try:
            bg.index.astype("int64")
            print("OK: l'index de bus_geodata peut etre converti en entiers.")
        except Exception as exc:
            print("Probleme probable: l'index de bus_geodata n'est pas numerique:", exc)

    print("\nVerification de parameters:")
    if "parameters" in xls:
        params = xls["parameters"]
        display(params.head(10))
        print("Colonnes lues dans parameters:", list(params.columns))
        if "parameter" not in params.columns:
            print("Probleme probable: la feuille parameters n'a pas de colonne appelee 'parameter'.")
            print("C'est la derniere erreur visible dans ton traceback: KeyError: 'parameter'.")
    else:
        print("Probleme probable: la feuille 'parameters' manque.")

    print("\nTest direct avec pandapower.from_excel:")
    try:
        net_from_excel = pp.from_excel(filename)
        print("OK: pandapower a charge le fichier.")
        print("Tables chargees:", list(net_from_excel.keys()))
    except Exception:
        print("Erreur pandapower complete:")
        traceback.print_exc()

diagnose_pandapower_excel(file_path)


In [ ]:
import pandas as pd
import pandapower as pp
from pandapower.io_utils import from_dict_of_dfs
from pandapower.convert_format import convert_format

file_path = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/1_0_grid.xlsx"

# Le fichier a ete cree avec pandapower 2.14.6, alors que l'environnement utilise 3.2.1.
# En pandapower 3.x, bus_geodata n'est plus dans le reseau vide par defaut.
xls = pd.read_excel(file_path, sheet_name=None, index_col=0, engine="openpyxl")

lv_grid = pp.create_empty_network()
lv_grid["bus_geodata"] = pd.DataFrame(columns=["x", "y", "coords"])

lv_grid = from_dict_of_dfs(xls, net=lv_grid)
convert_format(lv_grid)

pp.runpp(lv_grid, numba=False)

print("Converged:", lv_grid.converged)
print("Bus:", len(lv_grid.bus))
print("Lines:", len(lv_grid.line))
print("Loads:", len(lv_grid.load))
display(lv_grid.res_bus.head())


In [ ]:
import pandas as pd
import pandapower as pp
from pandapower.io_utils import from_dict_of_dfs
from pandapower.convert_format import convert_format

def from_excel_compatible(file_path):
    xls = pd.read_excel(file_path, sheet_name=None, index_col=0, engine="openpyxl")

    grid = pp.create_empty_network()
    if "bus_geodata" in xls and "bus_geodata" not in grid:
        grid["bus_geodata"] = pd.DataFrame(columns=xls["bus_geodata"].columns)

    grid = from_dict_of_dfs(xls, net=grid)
    convert_format(grid)
    return grid

file_path = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/1_0_grid.xlsx"

lv_grid = from_excel_compatible(file_path)
pp.runpp(lv_grid, numba=False)

print("Converged:", lv_grid.converged)
print(f"{len(lv_grid.bus)} bus, {len(lv_grid.line)} lignes, {len(lv_grid.load)} charges")
display(lv_grid.res_bus.head())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "nodes_valais_filtered.csv"

nodes = pd.read_csv(csv_path)
nodes = nodes.dropna(subset=["longitude", "latitude"])

print(f"Nombre de points affiches: {len(nodes)}")

plt.figure(figsize=(10, 8))
plt.scatter(nodes["longitude"], nodes["latitude"], s=12, alpha=0.7)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Nodes Valais filtres")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px

csv_path = "nodes_valais.csv"

nodes = pd.read_csv(csv_path)
nodes = nodes.dropna(subset=["longitude", "latitude", "grid_id"])

nodes_zoom = nodes[(nodes["longitude"] >= 7.95) & (nodes["longitude"] <= 8.00)]
grid_ids = sorted(nodes_zoom["grid_id"].dropna().unique())

grid_labels = (
    nodes_zoom.groupby("grid_id", as_index=False)
    .agg(
        longitude=("longitude", "mean"),
        latitude=("latitude", "mean"),
        nb_points=("grid_id", "size"),
    )
)

print(f"Nombre de points affiches: {len(nodes_zoom)}")
print(f"Nombre de grilles: {len(grid_ids)}")
print("Grid IDs:", grid_ids)

fig = px.scatter(
    grid_labels,
    x="longitude",
    y="latitude",
    text="grid_id",
    size="nb_points",
    hover_data=["grid_id", "nb_points"],
    title="Grilles Valais - zoom longitude 7.95 a 8.00",
)

fig.update_traces(textposition="top center")
fig.update_layout(
    width=900,
    height=700,
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    xaxis_range=[7.95, 8.00],
    yaxis_scaleanchor="x",
)

fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "nodes_valais.csv"

nodes = pd.read_csv(csv_path)
nodes = nodes.dropna(subset=["longitude", "latitude", "grid_id"])

nodes_87 = nodes[nodes["grid_id"].astype(str).str.startswith("87")]

print(f"Nombre de points avec grid_id 87: {len(nodes_87)}")
print("Grid IDs:", sorted(nodes_87["grid_id"].unique()))

plt.figure(figsize=(10, 8))
plt.scatter(nodes_87["longitude"], nodes_87["latitude"], s=14, alpha=0.75)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Nodes Valais - grid_id 87")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "nodes_valais.csv"

nodes = pd.read_csv(csv_path)
nodes = nodes.dropna(subset=["longitude", "latitude", "grid_id"])

grid_87_0 = nodes[nodes["grid_id"] == "87_0"]

print(f"Nombre de points dans la grille 87_0: {len(grid_87_0)}")

plt.figure(figsize=(10, 8))
plt.scatter(grid_87_0["longitude"], grid_87_0["latitude"], s=18, alpha=0.75)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Grille 87_0")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

nodes_path = "nodes_valais.csv"
edges_path = "edges_valais.csv"
grid_id = "87_0"

nodes = pd.read_csv(nodes_path)
edges = pd.read_csv(edges_path)

grid_nodes = nodes[nodes["grid_id"] == grid_id].dropna(subset=["osmid", "longitude", "latitude"])

def id_to_str(value):
    return str(int(value)) if float(value).is_integer() else str(value)

grid_nodes = grid_nodes.copy()
grid_nodes["node_id"] = grid_nodes["osmid"].apply(id_to_str)
node_coords = dict(zip(grid_nodes["node_id"], zip(grid_nodes["longitude"], grid_nodes["latitude"])))

edges = edges.copy()
edges["u_id"] = edges["u"].apply(id_to_str)
edges["v_id"] = edges["v"].apply(id_to_str)
candidate_edges = edges[edges["u_id"].isin(node_coords) & edges["v_id"].isin(node_coords)]

def parse_linestring(wkt):
    match = re.search(r"LINESTRING \((.*)\)", str(wkt))
    if not match:
        return []
    coords = []
    for point in match.group(1).split(","):
        lon, lat = point.strip().split()[:2]
        coords.append((float(lon), float(lat)))
    return coords

def close_xy(a, b, tol=2e-5):
    return abs(a[0] - b[0]) <= tol and abs(a[1] - b[1]) <= tol

grid_edges = []
for _, edge in candidate_edges.iterrows():
    coords = parse_linestring(edge["geometry"])
    if not coords:
        continue
    u_coord = node_coords[edge["u_id"]]
    v_coord = node_coords[edge["v_id"]]
    endpoints_match = (
        close_xy(coords[0], u_coord) and close_xy(coords[-1], v_coord)
    ) or (
        close_xy(coords[0], v_coord) and close_xy(coords[-1], u_coord)
    )
    if endpoints_match:
        grid_edges.append(coords)

print(f"Nombre de nodes dans {grid_id}: {len(grid_nodes)}")
print(f"Nombre d'edges dans {grid_id}: {len(grid_edges)}")

plt.figure(figsize=(10, 8))

for coords in grid_edges:
    lon, lat = zip(*coords)
    plt.plot(lon, lat, color="tab:blue", linewidth=1.1, alpha=0.7)

plt.scatter(
    grid_nodes["longitude"],
    grid_nodes["latitude"],
    s=22,
    color="tab:red",
    edgecolor="white",
    linewidth=0.4,
    zorder=3,
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Grille {grid_id} - nodes et edges")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

grid_file = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/87_0_grid.xlsx"
nodes_file = "nodes_valais.csv"
grid_id = "87_0"

bus = pd.read_excel(grid_file, sheet_name="bus", index_col=0, engine="openpyxl")
line = pd.read_excel(grid_file, sheet_name="line", index_col=0, engine="openpyxl")
nodes = pd.read_csv(nodes_file)

def id_to_str(value):
    return str(int(value)) if float(value).is_integer() else str(value)

def node_id_from_bus_name(name):
    match = re.search(r"_node_(.+)$", str(name))
    return match.group(1) if match else None

nodes_87_0 = nodes[nodes["grid_id"] == grid_id].dropna(subset=["osmid", "longitude", "latitude"]).copy()
nodes_87_0["node_id"] = nodes_87_0["osmid"].apply(id_to_str)
node_lookup = nodes_87_0.drop_duplicates("node_id").set_index("node_id")[["longitude", "latitude"]]

bus_plot = bus.copy()
bus_plot["bus_index"] = bus_plot.index
bus_plot["node_id"] = bus_plot["name"].apply(node_id_from_bus_name)
bus_plot = bus_plot.join(node_lookup, on="node_id")

missing_coords = bus_plot[bus_plot[["longitude", "latitude"]].isna().any(axis=1)]
if len(missing_coords):
    print("Bus sans coordonnees longitude/latitude:")
    display(missing_coords[["name", "node_id"]])

bus_coords = bus_plot.dropna(subset=["longitude", "latitude"]).set_index("bus_index")[["longitude", "latitude"]]

print(f"Fichier pandapower: {grid_file}")
print(f"Bus dans le fichier pandapower: {len(bus)}")
print(f"Lignes dans le fichier pandapower: {len(line)}")
print(f"Bus affiches avec coordonnees: {len(bus_coords)}")

plt.figure(figsize=(10, 8))

edges_drawn = 0
for _, edge in line.iterrows():
    from_bus = edge["from_bus"]
    to_bus = edge["to_bus"]
    if from_bus not in bus_coords.index or to_bus not in bus_coords.index:
        continue
    x = [bus_coords.loc[from_bus, "longitude"], bus_coords.loc[to_bus, "longitude"]]
    y = [bus_coords.loc[from_bus, "latitude"], bus_coords.loc[to_bus, "latitude"]]
    plt.plot(x, y, color="tab:blue", linewidth=1.1, alpha=0.75)
    edges_drawn += 1

plt.scatter(
    bus_coords["longitude"],
    bus_coords["latitude"],
    s=24,
    color="tab:red",
    edgecolor="white",
    linewidth=0.4,
    zorder=3,
)

print(f"Lignes affichees: {edges_drawn}")

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Pandapower grid 87_0 - bus et lignes")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()


In [ ]:
import pandapower as pp

grid_file = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/87_0_grid.xlsx"

grid_87_0 = pp.from_excel(grid_file)
pp.runpp(grid_87_0)

print("Converged:", grid_87_0.converged)
print("Bus:", len(grid_87_0.bus))
print("Lines:", len(grid_87_0.line))
print("Loads:", len(grid_87_0.load))

display(grid_87_0.res_bus.head())
display(grid_87_0.res_line.head())
